# Oracle CNE Fine-tuning with Unsloth
## Llama 3.1 8B on Google Colab Free Tier (T4 GPU)

This notebook fine-tunes Llama 3.1 8B on Oracle Cloud Native Environment (CNE) technical documentation using Unsloth for efficient training on limited GPU resources.

### Key Features:
- 🚀 Optimized for T4 GPU (16GB VRAM)
- 📊 285 Oracle CNE Q&A pairs
- ⚡ 4-bit quantization with QLoRA
- 💾 Memory-efficient training
- 📈 Built-in evaluation and monitoring

## Step 1: Install Dependencies

Install Unsloth and required packages. This is optimized for the free tier GPU.

In [ ]:
%%capture
import torch

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
%%capture
# Install Unsloth for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Load Model with 4-bit Quantization

We'll use Llama 3.1 8B Instruct with 4-bit quantization to fit in T4's 16GB VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model configuration for T4 GPU
max_seq_length = 2048  # Supports longer context for technical docs
dtype = None  # Auto-detect
load_in_4bit = True  # Essential for T4 GPU

# Load Llama 3.1 8B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Model loaded successfully!")

## Step 3: Configure LoRA Adapters

Add LoRA adapters for parameter-efficient fine-tuning.

In [ ]:
# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank - balance between performance and speed
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Optimized for 0 dropout
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory efficient
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✓ LoRA adapters configured!")

## Step 4: Upload and Prepare Training Data

Upload your `ocne_training_data.jsonl` file using the file upload button in the left sidebar, or run the cell below and use the file picker.

In [ ]:
from google.colab import files
import io

print("Please upload your ocne_training_data.jsonl file:")
uploaded = files.upload()

# Get the filename
training_file = list(uploaded.keys())[0]
print(f"✓ Uploaded: {training_file}")

In [ ]:
import json
from datasets import Dataset

# Load JSONL data
data = []
with open(training_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} training examples")

# Show sample
print("\nSample entry:")
print(f"Instruction: {data[0]['instruction'][:100]}...")
print(f"Response: {data[0]['response'][:100]}...")

## Step 5: Format Data with Llama 3.1 Chat Template

Convert Q&A pairs to Llama 3.1's chat format.

In [ ]:
# Llama 3.1 Instruct prompt template
llama_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert on Oracle Cloud Native Environment (CNE). Provide accurate, detailed technical answers based on official documentation.<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    responses = examples["response"]
    texts = []
    
    for instruction, response in zip(instructions, responses):
        text = llama_prompt.format(instruction, response) + EOS_TOKEN
        texts.append(text)
    
    return {"text": texts}

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(data)

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✓ Formatted {len(dataset)} examples")
print("\nSample formatted text:")
print(dataset[0]["text"][:500] + "...")

## Step 6: Split Dataset

Create train/validation split for monitoring.

In [ ]:
# Split dataset (90% train, 10% validation)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

## Step 7: Configure Training Parameters

Optimized for T4 GPU with 16GB VRAM.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Better for Q&A format
    args=TrainingArguments(
        # Training duration
        num_train_epochs=3,
        
        # Batch sizes (optimized for T4)
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 2*4 = 8
        
        # Learning rate
        learning_rate=2e-4,
        warmup_steps=5,
        
        # Optimization
        optim="adamw_8bit",  # Memory efficient optimizer
        weight_decay=0.01,
        lr_scheduler_type="linear",
        
        # Logging and evaluation
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        
        # Memory optimization
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        
        # Output
        output_dir="outputs",
        report_to="none",  # Disable wandb/tensorboard for simplicity
        
        # Reproducibility
        seed=3407,
    ),
)

print("✓ Trainer configured!")
print(f"\nTraining Configuration:")
print(f"  - Effective batch size: {2 * 4}")
print(f"  - Total training steps: ~{len(train_dataset) // (2 * 4) * 3}")
print(f"  - Epochs: 3")
print(f"  - Learning rate: 2e-4")

## Step 8: Monitor GPU Memory Before Training

In [ ]:
import torch

# Check GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max GPU memory: {max_memory} GB")
print(f"Current GPU memory reserved: {start_gpu_memory} GB")
print(f"Available for training: {max_memory - start_gpu_memory} GB")

## Step 9: Start Training! 🚀

This will take approximately 15-30 minutes on a T4 GPU depending on the dataset size.

In [ ]:
# Start training
print("Starting training...\n")
trainer_stats = trainer.train()

print("\n✓ Training complete!")

## Step 10: View Training Statistics

In [ ]:
# Show final stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\n{'='*50}")
print("TRAINING STATISTICS")
print(f"{'='*50}")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"Final training loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"\nGPU Memory Usage:")
print(f"  - Peak memory: {used_memory} GB")
print(f"  - Memory for training: {used_memory_for_lora} GB")
print(f"  - Percentage used: {used_percentage}%")
print(f"{'='*50}")

## Step 11: Test the Fine-tuned Model

Let's test the model with some Oracle CNE questions!

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def ask_oracle_cne(question):
    """Ask the fine-tuned model an Oracle CNE question."""
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert on Oracle Cloud Native Environment (CNE). Provide accurate, detailed technical answers based on official documentation.<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        use_cache=True
    )
    
    response = tokenizer.batch_decode(outputs)[0]
    
    # Extract just the assistant's response
    assistant_response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    assistant_response = assistant_response.split("<|eot_id|>")[0].strip()
    
    return assistant_response

print("✓ Inference mode enabled. Ready to answer questions!")

In [ ]:
# Test questions
test_questions = [
    "How do I get help with ocne command syntax?",
    "What is the default configuration file location?",
    "How do I set up command line completion for ocne?",
]

print("Testing the fine-tuned model:\n")
print("="*70)

for i, question in enumerate(test_questions, 1):
    print(f"\n[Test {i}]")
    print(f"Q: {question}")
    print(f"\nA: {ask_oracle_cne(question)}")
    print("\n" + "-"*70)

## Step 12: Interactive Testing

Ask your own Oracle CNE questions!

In [ ]:
# Interactive question answering
print("Ask Oracle CNE questions (type 'quit' to exit):\n")

while True:
    question = input("\nYour question: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question:
        continue
    
    print(f"\nAnswer: {ask_oracle_cne(question)}")
    print("-" * 70)

## Step 13: Save the Model

Save the fine-tuned model in multiple formats.

In [ ]:
# Save LoRA adapters (smallest, fastest)
model.save_pretrained("oracle_cne_lora_model")
tokenizer.save_pretrained("oracle_cne_lora_model")

print("✓ LoRA adapters saved to 'oracle_cne_lora_model/'")

In [ ]:
# Optional: Save merged 16-bit model (if you have enough space)
# WARNING: This requires ~16GB disk space

save_merged = False  # Set to True if you want to save the merged model

if save_merged:
    model.save_pretrained_merged(
        "oracle_cne_merged_16bit",
        tokenizer,
        save_method="merged_16bit"
    )
    print("✓ Merged 16-bit model saved to 'oracle_cne_merged_16bit/'")
else:
    print("Skipping merged model save (set save_merged=True to enable)")

In [ ]:
# Optional: Save 4-bit quantized model (smallest)
save_4bit = False  # Set to True if you want to save 4-bit model

if save_4bit:
    model.save_pretrained_merged(
        "oracle_cne_merged_4bit",
        tokenizer,
        save_method="merged_4bit"
    )
    print("✓ Merged 4-bit model saved to 'oracle_cne_merged_4bit/'")
else:
    print("Skipping 4-bit model save (set save_4bit=True to enable)")

## Step 14: Download Model Files

Download your trained model to use elsewhere.

In [ ]:
# Create a zip file of the LoRA model
!zip -r oracle_cne_lora_model.zip oracle_cne_lora_model/

print("✓ Model zipped!")
print("\nDownloading...")

from google.colab import files
files.download('oracle_cne_lora_model.zip')

print("\n✓ Download complete!")

## Step 15: Optional - Push to HuggingFace Hub

Share your model on HuggingFace (requires HF token).

In [ ]:
# Uncomment and configure if you want to push to HuggingFace

# from huggingface_hub import login
# 
# # Login to HuggingFace (you'll need to enter your token)
# login()
# 
# # Push to hub
# model.push_to_hub(
#     "your-username/oracle-cne-llama-3.1-8b",
#     token="your_hf_token_here"
# )
# tokenizer.push_to_hub(
#     "your-username/oracle-cne-llama-3.1-8b",
#     token="your_hf_token_here"
# )
# 
# print("✓ Model pushed to HuggingFace Hub!")

print("To push to HuggingFace, uncomment and configure the code above.")

## Summary

### What You've Accomplished:
- ✅ Loaded Llama 3.1 8B Instruct with 4-bit quantization
- ✅ Fine-tuned on 285 Oracle CNE Q&A pairs
- ✅ Optimized for T4 GPU (free tier)
- ✅ Tested the model with sample questions
- ✅ Saved the model for future use

### Model Files:
- **LoRA adapters**: `oracle_cne_lora_model/` (~100-200MB)
- **Merged models**: Optional, depending on what you saved

### Next Steps:
1. Test with more Oracle CNE questions
2. Adjust training parameters if needed (epochs, learning rate)
3. Add more training data to improve performance
4. Deploy the model using vLLM, Ollama, or other inference servers

### Tips for Better Results:
- If loss isn't decreasing, try more epochs or lower learning rate
- If model is overfitting, reduce epochs or add more data
- For better quality, consider using a larger model (70B) with more GPU
- Monitor validation loss to ensure the model generalizes well

Happy training! 🚀